# Analyse financière : les grandes banques américaines (2006 – 2015)

Analyse exploratoire (EDA) des cours de bourse de six banques américaines, de la crise financière de 2007-2008 jusqu'au 1er janvier 2016. L'objectif est de pratiquer **pandas** et la **visualisation de données**.

**Banques étudiées**

| Ticker | Banque |
|---|---|
| BAC | Bank of America |
| C | Citigroup |
| GS | Goldman Sachs |
| JPM | JPMorgan Chase |
| MS | Morgan Stanley |
| WFC | Wells Fargo |

### Utilisation sur Google Colab
1. Menu **Exécution → Tout exécuter** (ou `Ctrl + F9`).
2. Les données sont téléchargées depuis **Yahoo Finance** : Colab fournit la connexion Internet nécessaire.
3. Si le téléchargement échoue (limite de requêtes de Yahoo), attendez une minute puis relancez uniquement la cellule de la section 1.

### Ce qui change par rapport au notebook d'origine
| Notebook d'origine | Cette version |
|---|---|
| `pandas_datareader` avec la source `'google'` (arrêtée depuis des années) | `yfinance` |
| `.ix[...]` (supprimé de pandas) | `.loc[...]` |
| `sns.distplot` (obsolète) | `sns.histplot` |
| `cufflinks` (plus maintenu) | `plotly` (`express` et `graph_objects`) |

## 0. Installation et imports

In [ ]:
# Mise à jour de yfinance : les anciennes versions sont souvent bloquées par Yahoo.
# (Si Colab affiche un avertissement de redémarrage : Exécution > Redémarrer la session, puis relancez.)
!pip install -q -U yfinance

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import yfinance as yf
from IPython.display import display

sns.set_style('whitegrid')
%matplotlib inline

# Périodes réutilisées plus bas pour découper les séries temporelles
an2008 = slice('2008-01-01', '2008-12-31')
an2015 = slice('2015-01-01', '2015-12-31')

## 1. Récupérer les données

On télécharge les cours journaliers (Open, High, Low, Close, Volume) du **1er janvier 2006 au 1er janvier 2016** pour chaque banque, puis on les regroupe dans un seul DataFrame `bank_stocks` à colonnes hiérarchiques (*ticker* puis *type d'information*).

> **À savoir : prix ajustés des splits.** Yahoo Finance corrige les cours passés des opérations de regroupement/division d'actions. Le notebook d'origine utilisait des données Google *non ajustées* : Citigroup y affichait des cours d'environ 500 $ avant mai 2011, à cause du *reverse split* 1 pour 10 de cette date, ce qui créait deux « rendements » artificiels énormes. Avec les données Yahoo, cet artefact disparaît et les cours de Citigroup sont environ 10 fois plus petits avant 2011. Certaines réponses de la section « EDA » (le titre le plus risqué, par exemple) peuvent donc différer de celles du projet d'origine : c'est normal.

In [ ]:
# Liste des tickers (ordre alphabétique) et noms complets
tickers = ['BAC', 'C', 'GS', 'JPM', 'MS', 'WFC']
noms = {
    'BAC': 'Bank of America',
    'C': 'Citigroup',
    'GS': 'Goldman Sachs',
    'JPM': 'JPMorgan Chase',
    'MS': 'Morgan Stanley',
    'WFC': 'Wells Fargo',
}

# Bornes de la période (Yahoo exclut la date de fin : dernier jour récupéré = 31/12/2015)
debut, fin = '2006-01-01', '2016-01-01'

# Copie locale pour ne pas re-télécharger à chaque exécution (supprimée si la session Colab est réinitialisée)
CACHE = Path('bank_stocks_yahoo.pkl')


def telecharger(tickers, debut, fin, essais=3):
    # Télécharge tous les tickers d'un coup, avec quelques nouvelles tentatives.
    for n in range(1, essais + 1):
        try:
            brut = yf.download(tickers, start=debut, end=fin,
                               auto_adjust=False,      # on garde Open/High/Low/Close bruts
                               group_by='ticker', progress=False, threads=False)
            if not brut.empty:
                return brut
            print(f'Essai {n}/{essais} : réponse vide de Yahoo Finance')
        except Exception as e:
            print(f'Essai {n}/{essais} : {type(e).__name__} : {e}')
        time.sleep(5 * n)
    raise RuntimeError("Impossible de télécharger les données Yahoo Finance. "
                       "Patientez une minute puis relancez cette cellule.")


if CACHE.exists():
    bank_stocks = pd.read_pickle(CACHE)
    print('Données chargées depuis la copie locale (supprimez', CACHE, 'pour re-télécharger).')
else:
    brut = telecharger(tickers, debut, fin)

    # Selon la version de yfinance, les niveaux sont (Ticker, Price) ou (Price, Ticker)
    if set(tickers) <= set(brut.columns.get_level_values(1)):
        brut = brut.swaplevel(axis=1)

    colonnes = ['Open', 'High', 'Low', 'Close', 'Volume']
    bank_stocks = pd.concat({t: brut[t][colonnes].dropna(how='all') for t in tickers}, axis=1)
    bank_stocks.index.name = 'Date'
    bank_stocks.to_pickle(CACHE)

# Noms des niveaux de colonnes
bank_stocks.columns.names = ['Bank Ticker', 'Stock Info']

print(f'{bank_stocks.shape[0]} jours de bourse, du {bank_stocks.index.min():%d/%m/%Y} '
      f'au {bank_stocks.index.max():%d/%m/%Y}')

Un DataFrame par banque, à partir de `bank_stocks`. Ils servent dans les graphiques plus bas.

In [ ]:
BAC = bank_stocks['BAC']
C = bank_stocks['C']
GS = bank_stocks['GS']
JPM = bank_stocks['JPM']
MS = bank_stocks['MS']
WFC = bank_stocks['WFC']

# Contrôle rapide : valeurs manquantes par colonne principale
bank_stocks.xs('Close', axis=1, level='Stock Info').isna().sum().to_frame('Valeurs manquantes (Close)').T

**Aperçu des premières lignes de `bank_stocks`**

In [ ]:
bank_stocks.head()

## 2. Analyse exploratoire (EDA)

### Quel est le cours de clôture maximal de chaque banque sur la période ?

In [ ]:
bank_stocks.xs(key='Close', axis=1, level='Stock Info').max()

### Les rendements journaliers

On crée un DataFrame `returns` contenant le rendement journalier de chaque action, défini par :

$$r_t = \frac{p_t - p_{t-1}}{p_{t-1}} = \frac{p_t}{p_{t-1}} - 1$$

La méthode `pct_change()` de pandas, appliquée à la colonne `Close`, calcule exactement cela.

In [ ]:
returns = pd.DataFrame()

for tick in tickers:
    returns[tick + ' Return'] = bank_stocks[tick]['Close'].pct_change()

returns.head()

### Pairplot des rendements

Quelle action se distingue, et pourquoi ? (La première ligne est vide car il n'y a pas de rendement pour le premier jour ; on l'exclut.)

In [ ]:
sns.pairplot(returns.dropna(), plot_kws={'s': 8, 'alpha': 0.5})
plt.show()

### Meilleure et pire journée de chaque banque

Les dates des plus fortes baisses et des plus fortes hausses journalières de chaque titre.

In [ ]:
r = returns.rename(columns=lambda c: c.replace(' Return', ''))

resume = pd.DataFrame({
    'Pire jour': r.idxmin().dt.date,
    'Pire rendement (%)': (r.min() * 100).round(1),
    'Meilleur jour': r.idxmax().dt.date,
    'Meilleur rendement (%)': (r.max() * 100).round(1),
})
display(resume)

pires = r.idxmin().dt.date.value_counts()
print(f'Pire journée la plus partagée : {pires.index[0]} ({pires.iloc[0]} banque(s) sur {len(tickers)})')

Dans le projet d'origine, quatre banques (BAC, GS, JPM et WFC) partageaient leur pire journée : le **20 janvier 2009**, jour de l'investiture de Barack Obama, sur fond d'inquiétudes sur la solidité du secteur bancaire. Comparez avec le tableau ci-dessus.

### Citigroup : les plus gros mouvements

Dans le projet d'origine, la pire et la meilleure journée de Citigroup tombaient sur deux séances consécutives (6 et 9 mai 2011), à cause du *reverse split*. Avec des cours ajustés, cet artefact disparaît : regardons les plus gros mouvements restants et leurs dates (rappel : l'État américain a annoncé un plan de sauvetage de Citigroup le 23 novembre 2008).

In [ ]:
c = returns['C Return']
gros_mouvements = c.loc[c.abs().nlargest(6).index].sort_index()

(gros_mouvements * 100).round(1).rename('Rendement de Citigroup (%)').to_frame()

### Quelle action est la plus risquée ?

On utilise l'écart-type des rendements comme mesure de risque (volatilité), sur toute la période puis sur la seule année 2015.

In [ ]:
risque = pd.DataFrame({
    'Toute la période': returns.std(),
    '2015': returns.loc[an2015].std(),
})
risque.index = risque.index.str.replace(' Return', '', regex=False)
display(risque.round(4))

print('Plus risquée sur toute la période :', risque['Toute la période'].idxmax())
print('Plus risquée en 2015              :', risque['2015'].idxmax())

### Distribution des rendements de Morgan Stanley en 2015

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(returns.loc[an2015, 'MS Return'], bins=100, kde=True, color='green')
plt.title('Morgan Stanley : distribution des rendements journaliers en 2015')
plt.xlabel('Rendement journalier')
plt.show()

### Distribution des rendements de Citigroup en 2008

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(returns.loc[an2008, 'C Return'], bins=100, kde=True, color='red')
plt.title('Citigroup : distribution des rendements journaliers en 2008')
plt.xlabel('Rendement journalier')
plt.show()

## 3. Visualisations

### Cours de clôture de chaque banque (graphique statique)

In [ ]:
close = bank_stocks.xs(key='Close', axis=1, level='Stock Info')

fig, ax = plt.subplots(figsize=(12, 5))
for tick in tickers:
    close[tick].plot(ax=ax, label=tick)
ax.set_title('Cours de clôture des banques (2006-2015)')
ax.set_ylabel('Prix de clôture ($)')
ax.legend()
plt.show()

### Le même graphique en version interactive (plotly)

Survolez les courbes pour lire les valeurs, zoomez en sélectionnant une zone, double-cliquez pour revenir à la vue d'ensemble, cliquez dans la légende pour masquer une banque.

In [ ]:
close_long = (close.rename_axis('Date')
                   .reset_index()
                   .melt(id_vars='Date', var_name='Banque', value_name='Prix de clôture ($)'))

fig = px.line(close_long, x='Date', y='Prix de clôture ($)', color='Banque',
              title='Cours de clôture des banques (2006-2015)')
fig.update_layout(hovermode='x unified', height=500)
fig.show()

### Moyennes mobiles

Moyenne mobile sur 30 jours comparée au cours de clôture de Bank of America en 2008. (La moyenne est calculée sur tout l'historique puis découpée sur 2008, pour ne pas avoir de trou en début d'année.)

In [ ]:
bac_close = BAC['Close']

plt.figure(figsize=(12, 6))
bac_close.loc[an2008].plot(label='BAC Close')
bac_close.rolling(window=30).mean().loc[an2008].plot(label='Moyenne mobile 30 jours')
plt.title('Bank of America en 2008 : cours de clôture et moyenne mobile 30 jours')
plt.ylabel('Prix ($)')
plt.legend()
plt.show()

### Corrélation entre les cours de clôture

In [ ]:
close_corr = close.corr()

plt.figure(figsize=(7, 6))
sns.heatmap(close_corr, annot=True, cmap='coolwarm')
plt.title('Corrélation des cours de clôture')
plt.show()

Un `clustermap` regroupe automatiquement les banques dont les cours évoluent de façon similaire :

In [ ]:
sns.clustermap(close_corr, annot=True, cmap='coolwarm')
plt.show()

La même matrice en version interactive :

In [ ]:
fig = px.imshow(close_corr, text_auto='.2f', color_continuous_scale='RdYlBu',
                title='Corrélation des cours de clôture')
fig.update_layout(width=600, height=550)
fig.show()

## 4. Analyse technique avec plotly

Le notebook d'origine utilisait la bibliothèque `cufflinks` (`iplot`, `ta_plot`), qui n'est plus maintenue et ne fonctionne plus avec les versions récentes de pandas et plotly. Les trois graphiques sont ici reconstruits avec `plotly.graph_objects`.

### Chandeliers japonais : Bank of America en 2015

Chaque bougie résume une journée : ouverture, plus haut, plus bas, clôture.

In [ ]:
bac_2015 = BAC.loc[an2015]

fig = go.Figure(go.Candlestick(
    x=bac_2015.index,
    open=bac_2015['Open'], high=bac_2015['High'],
    low=bac_2015['Low'], close=bac_2015['Close'],
    name='BAC',
))
fig.update_layout(title='Bank of America : chandeliers japonais (2015)',
                  xaxis_title='Date', yaxis_title='Prix ($)',
                  xaxis_rangeslider_visible=False, height=500)
fig.update_xaxes(rangebreaks=[dict(bounds=['sat', 'mon'])])  # masque les week-ends
fig.show()

### Moyennes mobiles simples (SMA) : Morgan Stanley en 2015

Moyennes mobiles sur 13, 21 et 55 jours. Quand une moyenne courte passe au-dessus d'une moyenne longue, on y lit souvent un signal haussier (et inversement).

In [ ]:
ms_close = MS['Close']

fig = go.Figure()
fig.add_trace(go.Scatter(x=ms_close.loc[an2015].index, y=ms_close.loc[an2015],
                         name='MS Close', line=dict(color='lightgray', width=1.5)))

for periode in (13, 21, 55):
    sma = ms_close.rolling(periode).mean().loc[an2015]
    fig.add_trace(go.Scatter(x=sma.index, y=sma, name=f'SMA {periode}'))

fig.update_layout(title='Morgan Stanley : moyennes mobiles simples (2015)',
                  xaxis_title='Date', yaxis_title='Prix ($)',
                  hovermode='x unified', height=500)
fig.show()

### Bandes de Bollinger : Bank of America en 2015

Les bandes de Bollinger entourent une moyenne mobile sur 20 jours de ± 2 écarts-types. Plus les bandes sont larges, plus le titre est volatil ; un cours qui touche la bande haute ou la bande basse s'écarte fortement de sa tendance récente.

In [ ]:
fenetre, nb_ecarts = 20, 2

milieu = bac_close.rolling(fenetre).mean()
ecart = bac_close.rolling(fenetre).std(ddof=0)
haut = milieu + nb_ecarts * ecart
bas = milieu - nb_ecarts * ecart

fig = go.Figure()
fig.add_trace(go.Scatter(x=haut.loc[an2015].index, y=haut.loc[an2015], name='Bande haute',
                         line=dict(color='gray', width=1)))
fig.add_trace(go.Scatter(x=bas.loc[an2015].index, y=bas.loc[an2015], name='Bande basse',
                         line=dict(color='gray', width=1),
                         fill='tonexty', fillcolor='rgba(128, 128, 128, 0.15)'))
fig.add_trace(go.Scatter(x=milieu.loc[an2015].index, y=milieu.loc[an2015], name=f'SMA {fenetre}',
                         line=dict(color='orange', width=1.5, dash='dash')))
fig.add_trace(go.Scatter(x=bac_close.loc[an2015].index, y=bac_close.loc[an2015], name='BAC Close',
                         line=dict(color='royalblue', width=2)))

fig.update_layout(title='Bank of America : bandes de Bollinger (2015)',
                  xaxis_title='Date', yaxis_title='Prix ($)',
                  hovermode='x unified', height=500)
fig.show()

## Terminé

Pour aller plus loin :
- Remplacer `tickers` par d'autres titres (en adaptant les variables `BAC`, `C`, … et les noms).
- Prolonger la période (`debut`, `fin`) : pensez à supprimer `bank_stocks_yahoo.pkl` pour forcer un nouveau téléchargement.
- Comparer le comportement des banques pendant la crise de 2008 et pendant la crise sanitaire de 2020.